# 유튜브 스크립트 불러오기

pip install langchain

In [3]:
from langchain_community.document_loaders import YoutubeLoader

pip install youtube_transcript_api

In [4]:
loader = YoutubeLoader.from_youtube_url("https://www.youtube.com/watch?v=Pn-W41hC764")
transcript = loader.load()
transcript

[Document(metadata={'source': 'Pn-W41hC764'}, page_content="I alluded in my opening remarks to the the jobs issue the economic effects on employment uh I think you have said uh in fact and I'm going to quote development of superhuman machine intelligence is probably the greatest threat to the continued existence of humanity end quote you may have had in mind the effect on on jobs which is really my biggest nightmare in the long term uh let me ask you uh what your biggest nightmare is and whether you share that concern like with all technological revolutions I expect there to be significant impact on jobs but exactly what that impact looks like is very difficult to predict if we went back to the the other side of a previous technological Revolution talking about the jobs that exist on the other side um you know you can go back and read books of this it's what people said at the time it's difficult I believe that there will be far greater jobs on the other side of this and the jobs of to

# 긴 내용 요약하기

In [5]:
from langchain.prompts import PromptTemplate
from langchain.chains.summarize import load_summarize_chain
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chat_models import ChatOpenAI

## 텍스트 쪼개기

In [6]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=4000, chunk_overlap=0)
text = text_splitter.split_documents(transcript)

In [7]:
len(text)

2

In [8]:
text[0]

Document(metadata={'source': 'Pn-W41hC764'}, page_content="I alluded in my opening remarks to the the jobs issue the economic effects on employment uh I think you have said uh in fact and I'm going to quote development of superhuman machine intelligence is probably the greatest threat to the continued existence of humanity end quote you may have had in mind the effect on on jobs which is really my biggest nightmare in the long term uh let me ask you uh what your biggest nightmare is and whether you share that concern like with all technological revolutions I expect there to be significant impact on jobs but exactly what that impact looks like is very difficult to predict if we went back to the the other side of a previous technological Revolution talking about the jobs that exist on the other side um you know you can go back and read books of this it's what people said at the time it's difficult I believe that there will be far greater jobs on the other side of this and the jobs of tod

In [9]:
text[1]

Document(metadata={'source': 'Pn-W41hC764'}, page_content="the less you can get a proper read on that so it's important first of all that scientists be part of that process and second that we have much greater transparency about what actually goes into these systems if we don't know what's in them then we don't know exactly how well they're doing when we give something new and we don't know how good a benchmark that will be for something that's entirely novel so I could go into that more but I want to flag that second is on jobs past performance history is not a guarantee of the future it has always been the case in the past that we have had more jobs that new jobs new professions come in as new technologies come in I think this one's going to be different and the real question is over what time time scale is it going to be 10 years is it going to be 100 years and I don't think anybody knows the answer to that question I think in the long run so-called artificial general intelligence r

## 사용할 LLM 모델 설정

pip install openai
pip install tiktoken

In [ ]:
llm = ChatOpenAI(temperature=0,
                    openai_api_key="API_key",
                    max_tokens=3000,
                    model_name="gpt-3.5-turbo",
                    request_timeout=120
                )

## 요약에 사용할 프롬프트

In [15]:
# 각각의 chunck 를 요약하기
prompt = PromptTemplate(
    template="""Summarize the youtube video whose transcript is provided within backticks \
    ```{text}```
    """, input_variables=["text"]
)

# 요약된 내용들을 취합하여 다시한번 요약하기
combine_prompt = PromptTemplate(
    template="""Combine all the youtube video transcripts provided within backticks \
    ```{text}```
    Provide a concise summary between 8 to 10 sentences.
    """, input_variables=["text"]
)

## 요약 시작하기

In [16]:
chain = load_summarize_chain(llm, chain_type="map_reduce", verbose=False, map_prompt=prompt, combine_prompt=combine_prompt)

In [17]:
output = chain.run(text)

In [18]:
output

'The video discusses the impact of superhuman machine intelligence on jobs and the economy, highlighting the potential for automation to eliminate some jobs while creating new and improved opportunities. It stresses the importance of preparing the workforce to collaborate with AI technologies and emphasizes the need for transparency and clear guidelines in AI models. Scientists are urged to play a key role in AI system development to ensure understanding and accountability. The potential effects of artificial general intelligence on future job prospects are also explored, with concerns raised about potential harm from AI technology. Despite risks, speakers express optimism in human creativity and adaptability to new technologies. Overall, the message is that while AI offers significant benefits, careful management is essential to mitigate potential risks.'

## 번역하기

In [22]:
from deep_translator import GoogleTranslator

def google_trans(messages):
    return GoogleTranslator(source='auto', target='ko').translate(messages)

result = google_trans(output)
print(result)

이 비디오는 Super -Human Machine Intelligence가 직업과 경제에 미치는 영향에 대해 논의하여 새롭고 개선 된 기회를 창출하면서 일부 작업을 제거 할 수있는 자동화의 잠재력을 강조합니다. 그것은 AI 기술과 협력 할 수 있도록 인력을 준비하는 것의 중요성을 강조하고 AI 모델의 투명성 및 명확한 지침의 필요성을 강조합니다. 과학자들은 이해와 책임을 보장하기 위해 AI 시스템 개발에서 핵심적인 역할을하도록 촉구합니다. AI 기술의 잠재적 인 피해에 대한 우려가 제기되면서 미래의 직무 전망에 대한 인공 일반 정보의 잠재적 영향도 탐구됩니다. 위험에도 불구하고, 화자는 인간의 창의성과 새로운 기술에 대한 적응성에 대한 낙관론을 표현합니다. 전반적으로 AI는 상당한 이점을 제공하지만 잠재적 위험을 완화하는 데 신중한 관리가 필수적이라는 메시지입니다.
